1. Libraries and initialising 
Change file path here

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import savgol_filter
from pyproj import Transformer
import requests
import rasterio
import io
import math
import time

IMU_FILE_PATH = r"D:\Data_gathered\2026_05_01\Camera\14_30_00\back_01_05_2026-14_30_00.mcap"
GPS_FILE_PATH = r"D:\Data_gathered\2026_05_01\Rosbag\14_30_00\rosbag\rosbag_0.mcap"

Reading data

In [ ]:
GPS_LAT_LON_TOPIC = "/navsat_topic"
#TODO :Check if time is correct
IMU_TOPICS = "/zed/pose"

# ------------------- DATA CONTAINERS -----------------------
data = {"t": [], "x": [], "y": [], "z": []}
gps_data = {"t": [], "v": []}
gps_coords = {"t": [], "lat": [], "lon": []}

# ------------------- FILE READING --------------------------
print("Reading IMU and GPS files...")
# (IMU Reading)
with open(IMU_FILE_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[IMU_TOPICS]):
        if channel.topic == IMU_TOPICS:
            data["t"].append(ros_msg.header.stamp.nanosec / 1e9 + ros_msg.header.stamp.sec)
            data["x"].append(ros_msg.pose.position.x)
            data["y"].append(ros_msg.pose.position.y)
            data["z"].append(ros_msg.pose.position.z)
print("Done with IMU")
# (GPS Reading)

with open(GPS_FILE_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[GPS_LAT_LON_TOPIC]):
        # Get Coordinates
        if channel.topic == GPS_LAT_LON_TOPIC:
            gps_coords["t"].append(message.publish_time / 1e9)
            gps_coords["lat"].append(ros_msg.latitude)
            gps_coords["lon"].append(ros_msg.longitude)
print("Finished reading files")


Helper functions

In [ ]:
def quaternion_to_pitch(qx, qy, qz, qw): #TODO KNOW THIS
    qx, qy, qz, qw = np.array(qx), np.array(qy), np.array(qz), np.array(qw)
    sinp = 2 * (qw * qy - qz * qx)
    sinp = np.clip(sinp, -1, 1)
    return np.degrees(np.arcsin(sinp))

def get_dtm_elevation(x, y, session):
    """Fetches AHN4 DTM elevation for a single Dutch RD coordinate."""
    wcs_url = "https://service.pdok.nl/rws/ahn/wcs/v1_0"
    params = {
        "service": "WCS", "version": "1.0.0", "request": "GetCoverage",
        "coverage": "dtm_05m", "crs": "EPSG:28992",
        "bbox": f"{x-0.5},{y-0.5},{x+0.5},{y+0.5}",
        "width": 1, "height": 1, "format": "GEOTIFF"
    }
    try:
        resp = session.get(wcs_url, params=params, timeout=5)
        if resp.status_code == 200:
            with rasterio.open(io.BytesIO(resp.content)) as src:
                return src.read(1)[0][0]
    except Exception:
        pass
    return None

Height map processing

In [ ]:
print("Processing AHN4 True Ground Elevation...")

# Convert all GPS to RD coordinates
transformer = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
rd_xs, rd_ys = transformer.transform(gps_coords["lon"], gps_coords["lat"])
rd_xs = np.array(rd_xs)
rd_ys = np.array(rd_ys)

# Compute full bounding box with 1 m buffer
margin = 1.0
xmin = rd_xs.min() - margin
xmax = rd_xs.max() + margin
ymin = rd_ys.min() - margin
ymax = rd_ys.max() + margin

# Grid dimensions at 0.5 m AHN4 resolution
res = 0.5
gw = int(math.ceil((xmax - xmin) / res)) + 1
gh = int(math.ceil((ymax - ymin) / res)) + 1

print(f"Fetching AHN4 bbox: ({xmin:.1f}, {ymin:.1f}) -> ({xmax:.1f}, {ymax:.1f}), grid {gw}x{gh}...")

wcs_url = "https://service.pdok.nl/rws/ahn/wcs/v1_0"
params = {
    "service": "WCS", "version": "1.0.0", "request": "GetCoverage",
    "coverage": "dtm_05m", "crs": "EPSG:28992",
    "bbox": f"{xmin},{ymin},{xmax},{ymax}",
    "width": gw, "height": gh, "format": "GEOTIFF",
}
resp = requests.get(wcs_url, params=params, timeout=60)
print(f"Response: {resp.status_code}")

with rasterio.open(io.BytesIO(resp.content)) as src:
    ahn4_data = src.read(1).astype(float)
    ahn4_transform = src.transform
    ahn4_nodata = src.nodata

if ahn4_nodata is not None:
    ahn4_data[ahn4_data == ahn4_nodata] = np.nan

# Sample elevation every 1 m along the GPS route
gps_cum_dist = np.concatenate([[0], np.cumsum(np.hypot(np.diff(rd_xs), np.diff(rd_ys)))])
sample_dists = np.arange(0, gps_cum_dist[-1], 1.0)
sample_xs = np.interp(sample_dists, gps_cum_dist, rd_xs)
sample_ys = np.interp(sample_dists, gps_cum_dist, rd_ys)

from rasterio.transform import rowcol
rows, cols = rowcol(ahn4_transform, sample_xs, sample_ys)
rows = np.clip(rows, 0, ahn4_data.shape[0] - 1)
cols = np.clip(cols, 0, ahn4_data.shape[1] - 1)

dtm_elevations = ahn4_data[rows, cols]
dtm_distances  = sample_dists
print(f"Done. Sampled {int(np.sum(~np.isnan(dtm_elevations)))} valid elevation points.")


IMU PROCESSING


In [ ]:
d = data[IMU_TOPICS[0]]
raw_t = np.array(d["t"])
t_plot = raw_t - raw_t[0]

t_gps_raw = np.array(gps_data["t"])
v_gps = np.array(gps_data["v"])


gps_dist_steps = np.sqrt(np.diff(rd_xs)**2 + np.diff(rd_ys)**2) # Calculate distance steps from GPS coordinates
gps_cum_dist = np.insert(np.cumsum(gps_dist_steps), 0, 0.0) # Start at 0m

imu_distance = np.interp(raw_t, gps_coords["t"], gps_cum_dist) # Interpolate GPS cumulative distance at IMU timestamps
ds = np.diff(imu_distance, prepend=0)
# Pitch calculation
pitch = quaternion_to_pitch(d["qx"], d["qy"], d["qz"], d["qw"])
pitch_bias = np.mean(pitch)
print(f"Calculated pitch bias: {pitch_bias:.2f} degrees")

pitch = (pitch - pitch_bias) # Remove bias (and invert sign) to match DTM convention
#pitch = savgol_filter(pitch, window_length=101, polyorder=3) 


# IMU Estimated Elevation
dt = np.diff(t_plot, prepend=0)
imu_elevation = np.cumsum(ds * np.sin(np.radians(pitch)))
# Align IMU elevation to start at the exact same NAP height as the DTM
if len(dtm_elevations) > 0:
    imu_elevation = imu_elevation + dtm_elevations[0]

PLOT

In [ ]:
# --- PLOTTING ---
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
#ax.plot(imu_distance, imu_elevation, label="IMU Estimated Elevation", color='blue', linewidth=2)

# Plot DTM Ground Truth
if len(dtm_distances) > 0:
    ax.plot(dtm_distances, dtm_elevations, label="AHN4 DTM (True Ground)", color='green', linewidth=2, linestyle="--")

ax.set_title("Ground Profile: IMU Estimate vs AHN4 True DTM", fontsize=14, fontweight="bold")
ax.set_xlabel("Distance Traveled (m)")
ax.set_ylabel("Elevation (m NAP)")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

## FROM HERE

In [ ]:
z_arr = np.array(data["z"])
t_arr = np.array(data["t"])
mask = (z_arr >= 14) & (z_arr <= 25)

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.plot(t_arr[mask], z_arr[mask], label="IMU Z Position", color='blue', linewidth=2)
ax.set_title("IMU Z Position Over Time", fontsize=14, fontweight="bold")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Z Position (m)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
TIME = 1777638690.249665836
DELTA_M = 0.1                   # sampling interval in meters

In [ ]:
t = np.array(data["t"])
x = np.array(data["x"])
y = np.array(data["y"])
z = np.array(data["z"])
ref_idx = np.argmin(np.abs(t - TIME)) #argmin returns the index of the minimum value in the array, which corresponds to the closest timestamp to TIME
dt = abs(data["t"][ref_idx] - TIME)
print(f"Using TIME = {TIME}, closest GPS message Δt = {dt:.3f}s " f"(index {ref_idx})")


s = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
cum_s = np.concatenate([[0], np.cumsum(s)])
cum_s_from_s0 = cum_s - cum_s[ref_idx]
idx_25m = np.argmax(cum_s_from_s0 >= 20.0)
idx_5m = np.argmin(cum_s_from_s0 <= -5.0)
mask = (t >= t[idx_5m]) & (t <= t[idx_25m])
s_rel = cum_s_from_s0[mask]
z_rel = z[mask] - z[ref_idx]
t_rel = t[mask]

plt.figure(figsize=(10, 5))
plt.plot(s_rel, z_rel, label="IMU Z relative to 20m point")
plt.xlabel("Distance from 20m point (m)")
plt.ylabel("Z relative to 20m point (m)")   
plt.title("IMU Z Position Relative to 20m Point")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
gps_t = np.array(gps_coords["t"])
ref_idx = np.argmin(np.abs(gps_t - TIME))
dt_err = abs(gps_t[ref_idx] - TIME)
print(f"Using TIME={TIME}, closest GPS dt={dt_err:.3f}s (index {ref_idx})")

# Cumulative distance along full GPS route
gps_cum_dist_ref = np.concatenate([[0], np.cumsum(np.hypot(np.diff(rd_xs), np.diff(rd_ys)))])
s0 = gps_cum_dist_ref[ref_idx]

# Regular grid from -5 m to +20 m around reference point
s_query = np.arange(-5.0, 20.0 + DELTA_M, DELTA_M)
s_abs   = s0 + s_query

# Interpolate RD position at each query distance
q_xs = np.interp(s_abs, gps_cum_dist_ref, rd_xs)
q_ys = np.interp(s_abs, gps_cum_dist_ref, rd_ys)

# Sample AHN4 from cached raster
from rasterio.transform import rowcol
rows_q, cols_q = rowcol(ahn4_transform, q_xs, q_ys)
rows_q = np.clip(rows_q, 0, ahn4_data.shape[0] - 1)
cols_q = np.clip(cols_q, 0, ahn4_data.shape[1] - 1)
gps_ahn4_elev = ahn4_data[rows_q, cols_q]

plt.figure(figsize=(10, 5))
plt.plot(s_query, gps_ahn4_elev, label="AHN4 Ground Elevation", color="green", linewidth=2)
plt.xlabel("Distance from reference (m)")
plt.ylabel("Elevation (m NAP)")
plt.title("AHN4 Ground Profile: -5 m to +20 m from timestamp")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Save output for validation

In [ ]:
#IMU
# ============================================================
import os

ROSBAG_FILE_PATH = GPS_FILE_PATH
USING_MCAP = True 

t = TIME # ref time
method = "IMU_validation"

if USING_MCAP:
    # Mirror the input folder structure
    gps_parts = ROSBAG_FILE_PATH.replace("/", os.sep).split(os.sep)
    date_folder = gps_parts[-5]
    time_folder = gps_parts[-3]

SAVE_DIR = os.path.join(r"D:\Validation_results", date_folder, time_folder)
os.makedirs(SAVE_DIR, exist_ok=True)

# -- Optional extra fields to include alongside the core variables --
extras = {
}

np.savez_compressed(
    os.path.join(SAVE_DIR, f"{method}_{t}.npz"),
    x      = None, # x coordinates in bike-local frame (x=0 at bike)
    y      = None,              # y coordinates in bike-local frame
    z      = z_rel, # z coordinates in bike-local frame
    s      = s_rel, # path length along the strip
    t      = [TIME], # reference time of the profile, in UNIX timestamp
    method = np.array([method]),
    **extras,
)
fpath = os.path.join(SAVE_DIR, f"{method}_{t}.npz")
print(f"Saved to {fpath}")
print(f"  Core   : x, y, z, s, t, method")
print(f"  Extras : {list(extras.keys())}")


In [ ]:
# ============================================================
# Save results to external drive
# ============================================================
import os

ROSBAG_FILE_PATH = GPS_FILE_PATH
USING_MCAP = True 

t = TIME # ref time
method = "GPS_AHN4_validation"

if USING_MCAP:
    # Mirror the input folder structure
    gps_parts = ROSBAG_FILE_PATH.replace("/", os.sep).split(os.sep)
    date_folder = gps_parts[-5]
    time_folder = gps_parts[-3]

SAVE_DIR = os.path.join(r"D:\Validation_results", date_folder, time_folder)
os.makedirs(SAVE_DIR, exist_ok=True)

# -- Optional extra fields to include alongside the core variables --
extras = {
}
np.savez_compressed(
    os.path.join(SAVE_DIR, f"{method}_{t}.npz"),
    x      = None, # x coordinates in bike-local frame (x=0 at bike)
    y      = None,              # y coordinates in bike-local frame
    z      = gps_ahn4_elev, # z coordinates in bike-local frame
    s      = s_query,           # path length along the strip
    t      = [TIME], # reference time of the profile, in UNIX timestamp
    method = np.array([method]),
    **extras,
)
fpath = os.path.join(SAVE_DIR, f"{method}_{t}.npz")
print(f"Saved to {fpath}")
print(f"  Core   : x, y, z, s, t, method")
print(f"  Extras : {list(extras.keys())}")
